# Description

Predicts drug-disease associations using the **module-based** approach: CLAMP-projected S-PrediXcan (disease, 49 tissues) and LINCS L1000 (drug) in the ARCHS4 latent space.

Mirrors `phenoplier/nbs/30_drug_disease_associations/100-lincs/011-prediction-gene_module_based.ipynb` exactly, using CLAMP (2366 LVs) instead of MultiPLIER (987 LVs).

For each of 49 tissues and 5 LV-count thresholds (all, 5, 10, 25, 50), runs:
$$\text{score} = -1 \times \mathbf{drug}^T \mathbf{disease}$$
in the LV space.

Saves 245 HDF5 files (49 tissues × 5 thresholds) with keys:
- `full_prediction`: all traits
- `prediction`: DOID-mapped (gold-standard comparison)
- `metadata`

# Modules loading

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd

from pyprojroot import here

# Settings

In [3]:
# If True, re-run even if output files already exist
FORCE_RUN = True

PREDICTION_METHOD = 'Module-based'

In [4]:
DATA_DIR = here('data/archs4/drug_diseases_associations')
display(DATA_DIR)
assert DATA_DIR.exists()

DRUG_DISEASE_DIR = here('output/drug_disease_analyses')

LINCS_DATA_DIR = DRUG_DISEASE_DIR / 'lincs'
display(LINCS_DATA_DIR)
assert LINCS_DATA_DIR.exists()

SPREDIXCAN_PROJ_DIR = DRUG_DISEASE_DIR / 'spredixcan' / 'proj'
display(SPREDIXCAN_PROJ_DIR)
assert SPREDIXCAN_PROJ_DIR.exists()

OUTPUT_PREDICTIONS_DIR = LINCS_DATA_DIR / 'predictions'
OUTPUT_PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
display(OUTPUT_PREDICTIONS_DIR)

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/data/archs4/drug_diseases_associations')

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/lincs')

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/spredixcan/proj')

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/lincs/predictions')

# Helper functions

In [5]:
def map_traits_to_doid(data, preferred_doids, ukb_efo, efo_xrefs, do_xrefs):
    """
    Maps trait columns (UKB full codes) to Disease Ontology IDs (DOID).
    For traits mapping to multiple DOIDs, prefers those in `preferred_doids`.
    When a DOID appears from multiple traits, keeps the maximum score.
    """
    doid_efo = efo_xrefs[efo_xrefs['target_id_type'] == 'DOID']

    trait_to_doid = {}
    for trait in data.columns:
        if trait not in ukb_efo.index:
            continue
        rows = ukb_efo.loc[trait]
        if isinstance(rows, pd.Series):
            rows = rows.to_frame().T

        all_efo_codes = set()
        for term_codes in rows['term_codes'].dropna():
            for code in str(term_codes).split(','):
                code = code.strip()
                if code:
                    all_efo_codes.add(code)

        all_doids = set()
        for efo_code in all_efo_codes:
            mask = doid_efo['term_id'] == efo_code
            all_doids.update(doid_efo[mask]['target_id'].values)
            if efo_code.startswith('EFO:'):
                efo_num = efo_code[4:]
                mask2 = (do_xrefs['resource'] == 'EFO') & (do_xrefs['resource_id'] == efo_num)
                all_doids.update(do_xrefs[mask2]['doid_code'].values)

        if not all_doids:
            continue

        preferred = sorted(all_doids & preferred_doids)
        trait_to_doid[trait] = preferred[0] if preferred else sorted(all_doids)[0]

    data_mapped = data.loc[:, list(trait_to_doid.keys())].rename(columns=trait_to_doid)
    data_mapped = data_mapped.T.groupby(level=0).max().T
    return data_mapped

In [6]:
def _zero_nontop_genes(trait_vector, n_top, use_abs=True):
    """Zeros all but the top `n_top` LV values in a Series."""
    values = trait_vector.abs() if use_abs else trait_vector
    top_idx = values.sort_values(ascending=False).head(n_top).index
    result = trait_vector.copy()
    result[~result.index.isin(top_idx)] = 0.0
    return result

In [7]:
def predict_dotprod_neg(
    drug_gene_data,
    gene_trait_data_filename,
    gene_trait_data,
    output_dir,
    base_method_name,
    preferred_doid_list,
    force_run,
    n_top_conditions=None,
    use_abs=True,
):
    """
    Computes drug-disease predictions as: score = -1 * drug^T * disease
    File naming matches PhenoPlier: {stem}-{all_genes|top_N_genes}-prediction_scores.h5
    Uses n_top_genes as metadata key (consistent with PhenoPlier convention).
    """
    output_subdir = output_dir / 'dotprod_neg'
    output_subdir.mkdir(exist_ok=True, parents=True)

    suffix = 'all_genes' if n_top_conditions is None else f'top_{n_top_conditions}_genes'
    stem = gene_trait_data_filename.stem
    output_file = output_subdir / f'{stem}-{suffix}-prediction_scores.h5'

    print(f'  predicting {suffix}...', end='')
    if output_file.exists() and not force_run:
        print('  already run')
        return
    print('')

    disease_data = gene_trait_data.copy()
    if n_top_conditions is not None:
        disease_data = disease_data.apply(
            lambda x: _zero_nontop_genes(x, n_top_conditions, use_abs)
        )

    # score = -1 * (LVs x drugs)^T dot (LVs x traits) → drugs x traits
    scores = -1.0 * drug_gene_data.T.dot(disease_data)
    print(f'    shape: {scores.shape}')

    with pd.HDFStore(output_file, mode='w', complevel=4) as store:
        # full prediction (all traits)
        scores.index.name = 'drug'
        scores.columns.name = 'trait'
        full_pred = (
            scores.unstack()
            .reset_index()
            .rename(columns={0: 'score'})
        )
        full_pred['trait'] = full_pred['trait'].astype('category')
        full_pred['drug'] = full_pred['drug'].astype('category')
        assert full_pred.shape == full_pred.dropna().shape
        store.put('full_prediction', full_pred, format='table')

        # DOID-mapped prediction for gold-standard comparison
        scores_doid = map_traits_to_doid(
            scores, preferred_doid_list, ukb_efo, efo_xrefs, do_xrefs
        )
        assert scores_doid.index.is_unique
        assert scores_doid.columns.is_unique

        scores_doid.index.name = 'drug'
        scores_doid.columns.name = 'trait'
        doid_pred = (
            scores_doid.unstack()
            .reset_index()
            .rename(columns={0: 'score'})
        )
        doid_pred['trait'] = doid_pred['trait'].astype('category')
        doid_pred['drug'] = doid_pred['drug'].astype('category')
        assert doid_pred.shape == doid_pred.dropna().shape
        store.put('prediction', doid_pred, format='table')

        # Use n_top_genes as key (matches PhenoPlier convention for both methods)
        meta = pd.DataFrame({
            'method': [base_method_name],
            'n_top_genes': [-1.0 if n_top_conditions is None else float(n_top_conditions)],
            'data': [stem],
        })
        store.put('metadata', meta, format='table')

    print(f'    saved to: {output_file}')

# Load PharmacotherapyDB gold standard

In [8]:
gold_standard = pd.read_pickle(DATA_DIR / 'gold_standard.pkl')
display(gold_standard.shape)
display(gold_standard.head())

doids_in_gold_standard = set(gold_standard['trait'])
print(f'Unique DOIDs in gold standard: {len(doids_in_gold_standard)}')

(998, 3)

,trait,drug,true_class
0,DOID:10652,DB00843,1
1,DOID:10652,DB00674,1
2,DOID:10652,DB01043,1
3,DOID:10652,DB00989,1
4,DOID:10652,DB00810,0


Unique DOIDs in gold standard: 87


# Load trait → DOID mapping files

In [9]:
ukb_efo = pd.read_csv(
    DATA_DIR / 'phenomexcan_traits_fullcode_to_efo.tsv',
    sep='\t',
    index_col='ukb_fullcode',
)
# PhenoPlier stores trait full codes with hyphens (e.g. "I70-Diagnoses_...") but
# our S-PrediXcan data uses underscores throughout (e.g. "I70_Diagnoses_...").
# Normalize the index so lookups work correctly.
ukb_efo.index = [idx.replace('-', '_', 1) for idx in ukb_efo.index]

efo_xrefs = pd.read_csv(DATA_DIR / 'term_id_xrefs.tsv.gz', sep='\t')
do_xrefs = pd.read_csv(DATA_DIR / 'xrefs-prop-slim.tsv', sep='\t')

# Load LINCS projection

In [10]:
input_file = LINCS_DATA_DIR / 'lincs-projection.pkl'
display(input_file)
lincs_projection = pd.read_pickle(input_file)
display(lincs_projection.shape)
display(lincs_projection.head())

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/lincs/lincs-projection.pkl')

(2366, 1170)

perturbagen,DB00014,DB00091,DB00121,DB00130,DB00131,DB00132,DB00136,DB00140,DB00146,DB00150,...,DB08995,DB09002,DB09004,DB09009,DB09010,DB09015,DB09019,DB09020,DB09022,DB09023
LV1,-0.003664,0.021802,0.050683,-0.023411,0.054818,0.022504,-0.016074,-0.007949,-0.012020,-0.022674,...,-0.051500,-0.039418,-0.021987,0.012250,0.009535,0.012377,0.013959,-0.209899,0.026002,0.009327
LV2,0.011797,0.058622,0.001288,-0.023493,0.009715,0.001838,-0.025821,-0.026784,-0.010831,0.000237,...,-0.003300,-0.000286,0.004965,0.011235,0.002291,-0.003366,-0.002029,-0.067440,0.005340,0.011255
LV3,-0.000841,-0.055107,-0.006764,0.027218,-0.012003,-0.010472,-0.021370,-0.007238,-0.005116,0.005954,...,-0.003723,0.009973,-0.007469,0.009228,-0.010771,-0.017754,0.012783,0.066089,0.008973,-0.011303
LV4,0.031992,-0.511423,-0.071332,-0.059371,-0.035587,-0.033645,-0.076674,0.042806,-0.099315,0.031507,...,0.051060,0.022656,0.054687,0.014682,-0.078138,-0.014696,0.015666,-0.238956,-0.018649,-0.047013
LV5,-0.032510,0.123403,0.005948,0.031886,-0.045581,-0.006246,0.092339,0.034634,0.111000,-0.025291,...,-0.020998,-0.019916,-0.030282,0.009627,0.031024,-0.017982,-0.019684,0.164316,0.012215,-0.027974


# Load S-PrediXcan projected files

In [11]:
spredixcan_file_list = sorted(
    f for f in SPREDIXCAN_PROJ_DIR.glob('*.pkl') if f.name.startswith('spredixcan-')
)
display(len(spredixcan_file_list))
assert len(spredixcan_file_list) == 49

49

In [12]:
display(pd.read_pickle(spredixcan_file_list[0]).head())

,100001_raw_Food_weight,100002_raw_Energy,100003_raw_Protein,100004_raw_Fat,100005_raw_Carbohydrate,100006_raw_Saturated_fat,100007_raw_Polyunsaturated_fat,100008_raw_Total_sugars,100009_raw_Englyst_dietary_fibre,100010_Portion_size,...,Z50_Diagnoses_main_ICD10_Z50_Care_involving_use_of_rehabilitation_procedures,Z51_Diagnoses_main_ICD10_Z51_Other_medical_care,Z52_Diagnoses_main_ICD10_Z52_Donors_of_organs_and_tissues,Z53_Diagnoses_main_ICD10_Z53_Persons_encountering_health_services_for_specifie_procedures_not_carried_out,Z71_Diagnoses_main_ICD10_Z71_Persons_encountering_health_services_for_other_counselling_and_medical_advice_not_elsewhere_classified,Z76_Diagnoses_main_ICD10_Z76_Persons_encountering_health_services_in_other_circumstances,Z80_Diagnoses_main_ICD10_Z80_Family_history_of_malignant_neoplasm,Z85_Diagnoses_main_ICD10_Z85_Personal_history_of_malignant_neoplasm,Z87_Diagnoses_main_ICD10_Z87_Personal_history_of_other_diseases_and_conditions,pgc_scz2
LV1,0.007814,0.006795,0.009803,0.009815,-0.000923,0.006906,0.006776,-0.003193,0.011421,-0.007093,...,-0.012337,0.025431,-0.011921,0.000598,0.004758,-0.001167,-0.013817,0.007047,0.008864,-0.010720
LV2,0.003927,-0.004023,-0.006011,-0.004055,0.005487,-0.008535,-0.004991,0.012470,0.000268,-0.002455,...,-0.016772,0.020958,0.012497,0.008288,-0.032657,-0.017039,-0.005396,-0.007000,0.013178,0.039881
LV3,0.004304,-0.009663,-0.000327,-0.012830,-0.009193,-0.015054,-0.004725,-0.008997,-0.002176,0.003662,...,-0.005429,0.021735,0.002303,0.000157,0.007639,0.001659,0.005253,0.005989,-0.025748,0.026067
LV4,0.001599,-0.026818,-0.017631,-0.031678,-0.022896,-0.032414,-0.025261,-0.010275,-0.006312,0.003726,...,-0.016317,-0.016585,-0.031226,-0.013200,-0.011370,0.027171,0.013420,0.019905,-0.009798,-0.033794
LV5,-0.017567,-0.019870,-0.004849,-0.012572,-0.008135,-0.001412,-0.017222,-0.006193,-0.008891,-0.025552,...,0.007184,0.003116,0.002319,0.004610,0.011100,0.005989,-0.002826,0.021399,-0.012578,0.006030


# Predict drug-disease associations

In [13]:
# LV thresholds: None = all LVs (same as PhenoPlier module-based thresholds)
N_TOP_LVS_LIST = [None, 5, 10, 25, 50]

for spredixcan_file in spredixcan_file_list:
    print(spredixcan_file.name)

    # Load CLAMP-projected tissue-specific S-PrediXcan
    tissue_proj = pd.read_pickle(spredixcan_file)
    print(f'  shape: {tissue_proj.shape}')

    for ntc in N_TOP_LVS_LIST:
        predict_dotprod_neg(
            lincs_projection,
            spredixcan_file,
            tissue_proj,
            OUTPUT_PREDICTIONS_DIR,
            PREDICTION_METHOD,
            doids_in_gold_standard,
            FORCE_RUN,
            n_top_conditions=ntc,
            use_abs=True,
        )

    print('')

spredixcan-mashr-zscores-Adipose_Subcutaneous-projection.pkl
  shape: (2366, 4091)
  predicting all_genes...
    shape: (1170, 4091)


    saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/lincs/predictions/dotprod_neg/spredixcan-mashr-zscores-Adipose_Subcutaneous-projection-all_genes-prediction_scores.h5
  predicting top_5_genes...


    shape: (1170, 4091)


    saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/lincs/predictions/dotprod_neg/spredixcan-mashr-zscores-Adipose_Subcutaneous-projection-top_5_genes-prediction_scores.h5
  predicting top_10_genes...


    shape: (1170, 4091)


...

    saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/lincs/predictions/dotprod_neg/spredixcan-mashr-zscores-Whole_Blood-projection-top_10_genes-prediction_scores.h5
  predicting top_25_genes...


    shape: (1170, 4091)


    saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/lincs/predictions/dotprod_neg/spredixcan-mashr-zscores-Whole_Blood-projection-top_25_genes-prediction_scores.h5
  predicting top_50_genes...


    shape: (1170, 4091)


    saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/lincs/predictions/dotprod_neg/spredixcan-mashr-zscores-Whole_Blood-projection-top_50_genes-prediction_scores.h5

